# Rotation Robustness: CNN vs GCNN (p4) vs GCNN (p4m)

Author: Kavin

This notebook answers one question:

> **Do group-equivariant networks (GCNN p4, p4m) degrade less than a plain CNN
> when test images are rotated by angles the network was never explicitly trained on?**

### Experimental design

| | Detail |
|---|---|
| **Training** | All three models trained once on standard CIFAR-10 (+ optional mild augmentation) |
| **Test rotations** | 0°, 45°, 60°, 90°, 120°, 150°, 180° applied to the **test set only** |
| **Metric** | Top-1 accuracy at each rotation angle |
| **Parameter budget** | Equalised across all three models |

### Why this matters (Cohen & Welling 2016)

- **CNN (Z²)** — equivariant only to translations; should degrade sharply as angle increases.
- **GCNN p4** — equivariant to 90° rotations; should be robust at 0°/90°/180° and partially robust in between.
- **GCNN p4m** — equivariant to 90° rotations + reflections; same robustness profile as p4 with additional mirror symmetry.

At non-equivariant angles (45°, 60°, 120°, 150°) **all** models lose some accuracy, but the GCNNs should lose far less than the CNN.

---
**Folder layout**
```
experiment/
├── cifar10_rotation_experiment.ipynb   ← this file
├── cnngeneral.py
├── gcnngeneral.py
├── gcnn_p4mgeneral.py
├── cnngeneral.pth
├── gcnngeneral.pth
└── gcnn_p4mgeneral.pth
```


## 1. Environment Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')


In [ ]:
# ── Install / reconcile dependencies 
# IMPORTANT: This cell must be run TWICE on first use.
#
# Why: escnn installs a numpy version whose compiled C extensions are
# incompatible with the numpy already loaded in the kernel. The only safe
# fix is to (1) install everything, (2) restart the kernel so Python loads
# a single consistent numpy from scratch, then (3) continue normally.
#
# Run 1 → installs packages + restarts kernel automatically.
# Run 2 → all imports succeed, prints 'Environment OK'.

import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('escnn', 'matplotlib', 'seaborn', 'numpy>=1.24,<2.0')
print('All packages installed/updated.')

check = subprocess.run(
    [sys.executable, '-c',
     'import numpy as np; np.random.RandomState(); print(np.__version__)'],
    capture_output=True, text=True
)

if check.returncode == 0:
    try:
        import numpy as _np
        _np.random.RandomState()
        import escnn, matplotlib, seaborn
        print(f'Environment OK — numpy {_np.__version__}, escnn {escnn.__version__}')
    except (ValueError, ImportError):
        print('Kernel has stale numpy in memory — restarting now.')
        print('>>> Re-run this cell once more after the kernel restarts. <<<')
        try:
            import IPython
            IPython.get_ipython().kernel.do_shutdown(restart=True)
        except Exception:
            import os, signal
            os.kill(os.getpid(), signal.SIGTERM)
else:
    print('numpy install issue:')
    print(check.stderr[-500:])


In [ ]:
if IN_COLAB:
    from google.colab import files
    print('Upload cnngeneral.py, gcnngeneral.py, gcnn_p4mgeneral.py')
    uploaded = files.upload()
    print(f'Uploaded: {list(uploaded.keys())}')
else:
    print('Local run — skipping Colab upload cell.')


In [ ]:
import math
import time
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')


## 2. Experiment Configuration

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
NUM_EPOCHS      = 30        # 30 quick / 150 thorough / 300 paper-quality
BATCH_SIZE      = 128
LEARNING_RATE   = 0.05
MOMENTUM        = 0.9
WEIGHT_DECAY    = 1e-4
LR_MILESTONES   = [int(NUM_EPOCHS * 0.50), int(NUM_EPOCHS * 0.75)]
LR_GAMMA        = 0.1

# ── Training augmentation ─────────────────────────────────────────────────
# NOTE: we intentionally do NOT include rotation in the training augmentation.
# The test rotations are the out-of-distribution challenge we want to measure.
USE_AUGMENTATION = True

TEST_ANGLES = [0, 45, 60, 90, 120, 150, 180]

MODELS_TO_TRAIN = ['cnn', 'gcnn_p4', 'gcnn_p4m']

SEED = 42

EXPERIMENT_DIR = Path('.').resolve()
RESULTS_FILE   = EXPERIMENT_DIR / 'rotation_results.json'

DISPLAY_NAMES = {
    'cnn'      : 'CNN (Z²)',
    'gcnn_p4'  : 'GCNN (p4)',
    'gcnn_p4m' : 'GCNN (p4m)',
}
COLORS = {
    'cnn'      : '#5470c6',
    'gcnn_p4'  : '#91cc75',
    'gcnn_p4m' : '#ee6666',
}
MARKERS = {
    'cnn'      : 'o',
    'gcnn_p4'  : 's',
    'gcnn_p4m' : '^',
}

print('Configuration loaded.')
print(f'  Epochs        : {NUM_EPOCHS}')
print(f'  Augmentation  : {USE_AUGMENTATION}')
print(f'  Test angles   : {TEST_ANGLES}')
print(f'  Models        : {MODELS_TO_TRAIN}')


## 3. Reproducibility

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

print(f'Seed set to {SEED}')
print(f'sys.path[0] = {sys.path[0]}')


## 4. Data Loaders

### Training data
Standard CIFAR-10 with optional horizontal-flip + crop augmentation.
**No rotation augmentation** — rotation robustness must come from the model's architecture, not from data.

### Test data (rotation)
We create a `RotatedCIFAR10` wrapper that applies a fixed rotation angle before
the standard normalisation. One loader is created per test angle.


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

if USE_AUGMENTATION:
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
else:
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])

DATA_ROOT = EXPERIMENT_DIR / 'data'
trainset  = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True,
                                          download=True, transform=train_transform)
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
print(f'Train samples : {len(trainset)}')

class RotatedCIFAR10(Dataset):
    """CIFAR-10 test set with a fixed rotation applied before normalisation.

    fill=CIFAR10_MEAN is used so that the corners introduced by rotation
    match the dataset mean rather than being black, which would unfairly
    penalise large-angle rotations.
    """
    def __init__(self, root, angle_deg: float):
        self.base = torchvision.datasets.CIFAR10(root=root, train=False,
                                                  download=True, transform=None)
        self.angle = angle_deg
        self.to_tensor = transforms.ToTensor()
        self.normalise = transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
        # fill value per channel (0-255 scale)
        self.fill = tuple(int(m * 255) for m in CIFAR10_MEAN)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if self.angle != 0:
            img = TF.rotate(img, self.angle,
                            interpolation=TF.InterpolationMode.BILINEAR,
                            fill=self.fill)
        img = self.normalise(self.to_tensor(img))
        return img, label


# Build one loader per test angle
test_loaders = {}
for angle in TEST_ANGLES:
    ds = RotatedCIFAR10(DATA_ROOT, angle)
    test_loaders[angle] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                                     num_workers=2, pin_memory=True)

print(f'Test angles   : {list(test_loaders.keys())}')
print(f'Test samples  : {len(test_loaders[0])} batches × {BATCH_SIZE}')

sample_img_pil, _ = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=False, transform=None)[42]

fill_pil = tuple(int(m * 255) for m in CIFAR10_MEAN)
n_angles = len(TEST_ANGLES)
fig, axes = plt.subplots(1, n_angles, figsize=(2.5 * n_angles, 2.8))
for ax, angle in zip(axes, TEST_ANGLES):
    if angle == 0:
        img_show = sample_img_pil
    else:
        img_show = TF.rotate(sample_img_pil, angle,
                             interpolation=TF.InterpolationMode.BILINEAR,
                             fill=fill_pil)
    ax.imshow(img_show)
    ax.set_title(f'{angle}°', fontsize=11)
    ax.axis('off')
fig.suptitle('Test image at each rotation angle', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Model Instantiation

In [ ]:
def import_cnn():
    from cnngeneral import Net
    return Net

def import_gcnn_p4():
    from gcnngeneral import GCNN
    return GCNN

def import_gcnn_p4m():
    from gcnn_p4mgeneral import GCNNP4M
    return GCNNP4M

def build_model(model_key: str) -> nn.Module:
    if model_key == 'cnn':
        return import_cnn()()
    elif model_key == 'gcnn_p4':
        return import_gcnn_p4()()
    elif model_key == 'gcnn_p4m':
        return import_gcnn_p4m()()
    else:
        raise ValueError(f'Unknown model key: {model_key}')

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Model parameter counts:')
print(f'{"Model":<20} {"Parameters":>12}')
print('-' * 34)
for key in MODELS_TO_TRAIN:
    try:
        m = build_model(key)
        n = count_parameters(m)
        print(f'{DISPLAY_NAMES[key]:<20} {n:>12,}')
        del m
    except (ModuleNotFoundError, ImportError) as e:
        print(f'{DISPLAY_NAMES[key]:<20}   [MISSING: {e}]')


## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total   += inputs.size(0)
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total   += inputs.size(0)
    return total_loss / total, 100.0 * correct / total


def train_model(model_key: str, device, num_epochs: int):
    """Train one model; evaluate on the 0° test set each epoch for learning curves."""
    print(f'\n{"="*60}')
    print(f'  Training {DISPLAY_NAMES[model_key]}')
    print(f'{"="*60}')

    set_seed(SEED)
    model = build_model(model_key).to(device)
    n_params = count_parameters(model)
    print(f'  Parameters: {n_params:,}')

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(),
                          lr=LEARNING_RATE, momentum=MOMENTUM,
                          weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer,
                                                milestones=LR_MILESTONES,
                                                gamma=LR_GAMMA)

    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    best_acc  = 0.0
    ckpt_path = EXPERIMENT_DIR / f'{model_key}.pth'

    t0 = time.time()
    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        te_loss, te_acc = evaluate(model, test_loaders[0], criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_loss'].append(te_loss)
        history['test_acc'].append(te_acc)

        if te_acc > best_acc:
            best_acc = te_acc
            torch.save(model.state_dict(), ckpt_path)

        if epoch == 1 or epoch % 5 == 0 or epoch == num_epochs:
            elapsed = time.time() - t0
            print(f'  Epoch [{epoch:>3}/{num_epochs}]  '
                  f'train_acc={tr_acc:.2f}%  test_acc(0°)={te_acc:.2f}%  '
                  f'best={best_acc:.2f}%  [{elapsed:.0f}s]')

    print(f'  → Best 0° accuracy: {best_acc:.2f}%  |  {ckpt_path}')
    history['best_test_acc'] = best_acc
    history['n_params']      = n_params
    return history

print('Training functions defined.')


## 7. Train All Models


In [ ]:
train_histories = {}

for model_key in MODELS_TO_TRAIN:
    try:
        history = train_model(model_key, DEVICE, NUM_EPOCHS)
        train_histories[model_key] = history
    except (ModuleNotFoundError, ImportError) as e:
        print(f'\nSkipping {model_key}: {e}')
    except Exception as e:
        print(f'\nError training {model_key}: {e}')
        raise

print('\nTraining complete.')


## 8. Rotation Robustness Evaluation

Load the best checkpoint for each model and evaluate accuracy at every test angle.
This is the core of the experiment.


In [ ]:
@torch.no_grad()
def eval_at_angle(model, angle: int, device) -> float:
    """Return top-1 accuracy (%) on the test set rotated by `angle` degrees."""
    model.eval()
    loader = test_loaders[angle]
    correct, total = 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        _, predicted = model(inputs).max(1)
        correct += predicted.eq(targets).sum().item()
        total   += inputs.size(0)
    return 100.0 * correct / total


rotation_results = {}   # {model_key: {angle: acc}}

for model_key in MODELS_TO_TRAIN:
    ckpt_path = EXPERIMENT_DIR / f'{model_key}.pth'
    if not ckpt_path.exists():
        print(f'No checkpoint for {model_key} — skipping.')
        continue

    try:
        model = build_model(model_key).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.eval()

        angle_accs = {}
        print(f'\n{DISPLAY_NAMES[model_key]}')
        print(f'  {"Angle":>8}  {"Accuracy":>10}')
        print(f'  {"-"*22}')
        for angle in TEST_ANGLES:
            acc = eval_at_angle(model, angle, DEVICE)
            angle_accs[angle] = acc
            bar = '█' * int(acc / 2)
            print(f'  {angle:>6}°   {acc:>8.2f}%  {bar}')

        rotation_results[model_key] = angle_accs
        del model

    except Exception as e:
        print(f'Error evaluating {model_key}: {e}')
        raise

# Merge with training histories and save
all_results = {}
for key in rotation_results:
    all_results[key] = {
        'rotation_accs' : rotation_results[key],
        'train_history' : train_histories.get(key, {}),
    }

with open(RESULTS_FILE, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nResults saved → {RESULTS_FILE}')


## 8b. (Optional) Reload Results Without Retraining

If you already have `rotation_results.json` from a previous run, execute this
cell instead of Cells 7–8 above.


In [ ]:
if RESULTS_FILE.exists():
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    # json keys are strings; convert angle keys back to int
    for key in all_results:
        if 'rotation_accs' in all_results[key]:
            all_results[key]['rotation_accs'] = {
                int(k): v for k, v in all_results[key]['rotation_accs'].items()
            }
    rotation_results = {k: v['rotation_accs'] for k, v in all_results.items()}
    train_histories  = {k: v.get('train_history', {}) for k, v in all_results.items()}
    print(f'Loaded results from {RESULTS_FILE}')
    for key in rotation_results:
        print(f'  {DISPLAY_NAMES.get(key, key)}:',
              {f"{a}°": f"{v:.1f}%" for a, v in rotation_results[key].items()})
else:
    print('No results file found. Run training + evaluation cells first.')


## 9. Results & Visualisations

### 9.1 Accuracy vs Rotation Angle  *(the key plot)*

This is the central result. Each line is one model; the x-axis is the test
rotation angle. Equivariant models should form a flatter line than the CNN.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for model_key, angle_accs in rotation_results.items():
    name   = DISPLAY_NAMES.get(model_key, model_key)
    color  = COLORS.get(model_key, None)
    marker = MARKERS.get(model_key, 'o')
    angles = sorted(angle_accs.keys())
    accs   = [angle_accs[a] for a in angles]
    ax.plot(angles, accs, marker=marker, linewidth=2.2, markersize=8,
            label=name, color=color)
    # Annotate each point
    for a, acc in zip(angles, accs):
        ax.annotate(f'{acc:.1f}', (a, acc),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=7.5, color=color)

ax.set_xlabel('Test Rotation Angle (degrees)', fontsize=12)
ax.set_ylabel('Top-1 Accuracy (%)', fontsize=12)
aug_tag = ' [trained w/ flip+crop aug]' if USE_AUGMENTATION else ' [no aug]'
ax.set_title(f'CIFAR-10 Rotation Robustness{aug_tag}', fontsize=13, fontweight='bold')
ax.set_xticks(TEST_ANGLES)
ax.set_xticklabels([f'{a}°' for a in TEST_ANGLES])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# Shade p4 equivariant angles (multiples of 90°)
for a in [0, 90, 180]:
    if a in TEST_ANGLES:
        ax.axvline(a, color='#91cc75', linewidth=1, linestyle='--', alpha=0.4)
ax.text(0.5, 0.01, '  dashed lines = p4 equivariant angles (0°, 90°, 180°)',
        transform=ax.transAxes, fontsize=8, color='#555', va='bottom')

plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / 'rotation_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rotation_accuracy.png')


### 9.2 Accuracy Drop Relative to 0°

How much accuracy does each model *lose* as rotation increases?
A flat line (near 0) means the model is robust to that angle.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for model_key, angle_accs in rotation_results.items():
    name   = DISPLAY_NAMES.get(model_key, model_key)
    color  = COLORS.get(model_key, None)
    marker = MARKERS.get(model_key, 'o')
    angles = sorted(angle_accs.keys())
    baseline = angle_accs[0]
    drops  = [angle_accs[a] - baseline for a in angles]
    ax.plot(angles, drops, marker=marker, linewidth=2.2, markersize=8,
            label=name, color=color)
    for a, d in zip(angles, drops):
        ax.annotate(f'{d:+.1f}', (a, d),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=7.5, color=color)

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_xlabel('Test Rotation Angle (degrees)', fontsize=12)
ax.set_ylabel('Accuracy Change vs 0° (pp)', fontsize=12)
ax.set_title('Accuracy Drop Under Rotation  (0 = perfectly robust)',
             fontsize=13, fontweight='bold')
ax.set_xticks(TEST_ANGLES)
ax.set_xticklabels([f'{a}°' for a in TEST_ANGLES])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / 'rotation_drop.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rotation_drop.png')


### 9.3 Accuracy Heatmap

A compact view of all (model × angle) accuracy values.


In [ ]:
keys    = list(rotation_results.keys())
angles  = TEST_ANGLES
labels  = [DISPLAY_NAMES.get(k, k) for k in keys]
matrix  = np.array([[rotation_results[k][a] for a in angles] for k in keys])
col_labels = [f'{a}°' for a in angles]

fig, ax = plt.subplots(figsize=(len(angles) * 1.4 + 1, len(keys) * 1.1 + 1))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=max(0, matrix.min() - 5),
               vmax=min(100, matrix.max() + 2), aspect='auto')

ax.set_xticks(range(len(angles)))
ax.set_xticklabels(col_labels, fontsize=11)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_title('Accuracy (%) by Model and Rotation Angle',
             fontsize=13, fontweight='bold', pad=12)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f'{matrix[i, j]:.1f}%',
                ha='center', va='center', fontsize=10, fontweight='bold',
                color='black' if 30 < matrix[i, j] < 90 else 'white')

plt.colorbar(im, ax=ax, label='Accuracy (%)', fraction=0.03)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / 'rotation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rotation_heatmap.png')


### 9.4 Grouped Bar Chart — All Angles Side by Side


In [ ]:
keys   = list(rotation_results.keys())
n_models = len(keys)
n_angles = len(TEST_ANGLES)
x = np.arange(n_angles)
width = 0.22
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

fig, ax = plt.subplots(figsize=(13, 5))

for i, model_key in enumerate(keys):
    accs = [rotation_results[model_key][a] for a in TEST_ANGLES]
    bars = ax.bar(x + offsets[i], accs, width,
                  label=DISPLAY_NAMES.get(model_key, model_key),
                  color=COLORS.get(model_key, None),
                  edgecolor='white', zorder=3)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f'{acc:.1f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([f'{a}°' for a in TEST_ANGLES], fontsize=11)
ax.set_xlabel('Test Rotation Angle', fontsize=12)
ax.set_ylabel('Top-1 Accuracy (%)', fontsize=12)
ax.set_title('CIFAR-10 Accuracy per Rotation Angle — All Models',
             fontsize=13, fontweight='bold')
ax.set_ylim(max(0, matrix.min() - 8), min(100, matrix.max() + 6))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / 'rotation_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rotation_bar.png')


### 9.5 Training Learning Curves (0° test accuracy)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Learning Curves (test evaluated at 0°)',
             fontsize=14, fontweight='bold')

for model_key, hist in train_histories.items():
    if not hist:
        continue
    name  = DISPLAY_NAMES.get(model_key, model_key)
    color = COLORS.get(model_key, None)
    ep    = range(1, len(hist.get('train_acc', [])) + 1)
    axes[0].plot(ep, hist.get('train_acc', []), linestyle='--', color=color, alpha=0.6)
    axes[0].plot(ep, hist.get('test_acc',  []), linestyle='-',  color=color, label=name)
    axes[1].plot(ep, hist.get('train_loss', []), linestyle='--', color=color, alpha=0.6)
    axes[1].plot(ep, hist.get('test_loss',  []), linestyle='-',  color=color, label=name)

for ax, ylabel, title in zip(axes,
    ['Accuracy (%)', 'Cross-Entropy Loss'],
    ['Accuracy (solid=test, dashed=train)', 'Loss (solid=test, dashed=train)']):
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    for ms in LR_MILESTONES:
        ax.axvline(ms, color='grey', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: learning_curves.png')


## 10. Summary Table

In [ ]:
print('\n' + '='*80)
header = f'{"Model":<18}' + ''.join(f'  {a}°'.rjust(8) for a in TEST_ANGLES) + f'  {"MaxDrop":>9}'
print(header)
print('='*80)

baseline_0 = rotation_results.get('cnn', {}).get(0, None)

for key in rotation_results:
    angle_accs = rotation_results[key]
    row = f'{DISPLAY_NAMES.get(key, key):<18}'
    for a in TEST_ANGLES:
        row += f'  {angle_accs.get(a, float("nan")):>7.2f}%'
    drop = min(angle_accs.get(a, 0) - angle_accs.get(0, 0) for a in TEST_ANGLES)
    row += f'  {drop:>+8.2f}pp'
    print(row)

print('='*80)
print('MaxDrop = worst accuracy drop (pp) relative to 0° — closer to 0 is better.')

if baseline_0 is not None and len(rotation_results) > 1:
    print()
    print('Accuracy gain of GCNN models over CNN at each angle:')
    cnn_accs = rotation_results.get('cnn', {})
    for key in rotation_results:
        if key == 'cnn':
            continue
        gains = {a: rotation_results[key].get(a, 0) - cnn_accs.get(a, 0)
                 for a in TEST_ANGLES}
        gain_str = '  '.join(f'{a}°: {g:+.1f}pp' for a, g in gains.items())
        print(f'  {DISPLAY_NAMES.get(key, key)}: {gain_str}')
